# Building a CNN from Scratch in PyTorch: Conv Layers, Pooling, BatchNorm, and CIFAR-10

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/pytorch_5)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q torch torchvision torchaudio

In [ ]:
import torch
import torch.nn as nn

# Single conv layer: 1 input channel → 8 filters, 3x3 kernels
conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1)

# Input: batch=1, channel=1, height=32, width=32
x = torch.randn(1, 1, 32, 32)
out = conv(x)

print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Kernel: {conv.weight.shape}")
print(f"Parameters: {sum(p.numel() for p in conv.parameters()):,}")

In [ ]:
import torch
import torch.nn as nn

def conv_output_size(input_size, kernel, stride, padding):
    return (input_size + 2 * padding - kernel) // stride + 1

print("Output sizes for 32×32 input:")
for k, s, p in [(3, 1, 1), (3, 2, 1), (5, 1, 2), (3, 1, 0)]:
    out = conv_output_size(32, k, s, p)
    print(f"  kernel={k}, stride={s}, padding={p}: output={out}×{out}")

In [ ]:
import torch
import torch.nn as nn

pool = nn.MaxPool2d(kernel_size=2, stride=2)
x = torch.randn(1, 8, 32, 32)
out = pool(x)

print(f"Before pooling: {x.shape}")
print(f"After pooling:  {out.shape}")
print(f"Reduction: {x.shape[2]}×{x.shape[3]} → {out.shape[2]}×{out.shape[3]}")

In [ ]:
import torch
import torch.nn as nn

bn = nn.BatchNorm2d(num_features=8)
x = torch.randn(16, 8, 32, 32)  # batch=16, channels=8, spatial=32×32
out = bn(x)

print(f"Input mean (channel 0, batch stats): {x[:, 0].mean():.4f}")
print(f"Output mean (channel 0):             {out[:, 0].mean():.4f}")
print(f"Output std (channel 0):              {out[:, 0].std():.4f}")
print(f"BatchNorm parameters: gamma={bn.weight.shape}, beta={bn.bias.shape}")

In [ ]:
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    """Conv → BN → ReLU → optional MaxPool"""
    def __init__(self, in_ch: int, out_ch: int, pool: bool = False):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class CIFAR10Net(nn.Module):
    def __init__(self, num_classes: int = 10):
        super().__init__()

        # Feature extractor
        self.features = nn.Sequential(
            ConvBlock(3, 32),           # 32×32 → 32×32, 32 filters
            ConvBlock(32, 64, pool=True),  # 32×32 → 16×16, 64 filters
            ConvBlock(64, 128),         # 16×16 → 16×16, 128 filters
            ConvBlock(128, 256, pool=True), # 16×16 → 8×8, 256 filters
            ConvBlock(256, 256),        # 8×8 → 8×8, 256 filters
            ConvBlock(256, 512, pool=True), # 8×8 → 4×4, 512 filters
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),    # 4×4 → 1×1 (global average pooling)
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = CIFAR10Net()
dummy = torch.randn(4, 3, 32, 32)
out = model(dummy)

print(f"Output shape: {out.shape}")
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# CIFAR-10 statistics for normalization
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Download CIFAR-10 (first run only)
train_ds = datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_transform)
val_ds   = datasets.CIFAR10(root='./data', train=False, download=True, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# (Re-use model and loaders from above)
model = CIFAR10Net().to(device)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=1e-3, epochs=30,
    steps_per_epoch=len(train_loader),
)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def train_epoch(model, loader, optimizer, criterion, scheduler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / len(loader), correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        total_loss += criterion(logits, y).item()
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return total_loss / len(loader), correct / total

# Train for a few epochs (full 30 epochs reaches ~83% val accuracy)
for epoch in range(3):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, scheduler, device)
    vl_loss, vl_acc = eval_epoch(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1:2d}: train={tr_loss:.3f}/{tr_acc:.3f} | val={vl_loss:.3f}/{vl_acc:.3f}")

In [ ]:
import torch
import matplotlib
matplotlib.use('Agg')  # non-interactive backend

# First conv layer filters: (32, 3, 3, 3) → 32 filters, 3 channels, 3x3
filters = model.features[0].block[0].weight.data.cpu()
print(f"First layer filters shape: {filters.shape}")
print(f"Filter stats: min={filters.min():.3f}, max={filters.max():.3f}, std={filters.std():.3f}")

# Check filter diversity — trained filters should have varied norms
norms = filters.view(32, -1).norm(dim=1)
print(f"Filter L2 norms: min={norms.min():.3f}, max={norms.max():.3f}")

In [ ]:
import torch
import torch.nn as nn

pool = nn.AdaptiveAvgPool2d(1)

x_small = torch.randn(1, 512, 4, 4)
x_large = torch.randn(1, 512, 8, 8)

print(f"Small input {x_small.shape} → {pool(x_small).shape}")
print(f"Large input {x_large.shape} → {pool(x_large).shape}")